In [7]:
pip install torch-lucent

In [8]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from lucent.optvis import render, param
from lucent.modelzoo import inceptionv1
from PIL import Image

In [9]:
# Save all files to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def load_model(device):
    model = inceptionv1(pretrained=True)
    model.to(device)
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


model = load_model(DEVICE)

def activation_maximization(
    model,
    layer_name,
    neuron_id,
    image_size=512,
    num_steps=1024,
):


    objective = f"{layer_name}:{neuron_id}"

    param_f = lambda: param.image(
        image_size,
        fft=True,
        decorrelate=True,
    )

    images = render.render_vis(
        model,
        objective,
        param_f=param_f,
        thresholds=(num_steps,),
        show_image=False,
        verbose=False,
    )

    return images[0][0]

Using device: cuda
Downloading: "https://github.com/ProGamerGov/pytorch-old-tensorflow-models/raw/master/inception5h.pth" to /root/.cache/torch/hub/checkpoints/inception5h.pth


100%|██████████| 27.0M/27.0M [00:00<00:00, 316MB/s]


In [11]:
#Configuration
LAYER = "mixed4d" #Enter the layer you want to work on
NEURONS = range(0,528) #Enter the neuron/unit range for visualizing their activation maximizations

DRIVE_PATH = "/content/drive/MyDrive/activation_max_results"

save_dir = os.path.join(DRIVE_PATH, LAYER)
os.makedirs(save_dir, exist_ok=True)

# Generate and Save Images
for id in NEURONS:
    img = activation_maximization(
        model,
        LAYER,
        id,
        image_size=512,
        num_steps=1024
    )

    # Convert to uint8 image
    img_uint8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)

    #Save to drive
    save_path = os.path.join(save_dir, f"{id}.jpg")
    Image.fromarray(img_uint8).save(save_path, quality=95)

print(f"Saved → {save_path}")

100%|██████████| 1024/1024 [01:52<00:00,  9.14it/s]

Saved → /content/drive/MyDrive/activation_max_results/mixed4d/527.jpg
